In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
model.encode("test")

array([ 1.1573429e-02,  2.5136208e-02, -3.6701903e-02,  5.9324864e-02,
       -7.1490146e-03, -4.1194160e-02,  7.7087417e-02,  3.7442498e-02,
        1.2449003e-02, -6.1176615e-03,  1.7034272e-02, -7.7015340e-02,
       -3.9419177e-04,  2.7909052e-02, -1.5989188e-02, -6.8275258e-02,
        8.8846739e-03, -2.0280721e-02, -8.0359936e-02, -1.3074120e-02,
       -4.1099951e-02, -2.5898073e-02, -2.6538625e-02,  3.3052299e-02,
       -2.2079093e-02,  2.1046152e-02, -5.7921976e-02,  3.2948777e-02,
        2.9707383e-02, -6.2248435e-02,  3.8787980e-02,  3.1990722e-02,
        1.5330759e-02,  4.5306984e-02,  5.3149376e-02,  1.3360680e-02,
        4.1224957e-02,  2.8142877e-02,  1.9398455e-02, -3.2523405e-03,
       -3.6123630e-03, -1.4286025e-01,  3.8071208e-02, -1.0916241e-02,
        2.6093960e-02,  4.1369975e-02, -1.6015815e-02,  5.3560127e-02,
       -5.6859441e-02,  1.2246755e-02, -3.4996647e-02, -3.9754186e-02,
       -4.6143018e-02, -3.9112363e-02, -1.8003656e-02,  2.1634296e-02,
      

In [14]:
v1 = "Fire"
dv1 = model.encode(v1)

In [12]:
v2 = "Water"
dv2 = model.encode(v2)

In [15]:
dv1.dot(dv2)

np.float32(0.42806008)

In [3]:
from ingest import load_faq_data

documents = load_faq_data()

In [4]:
documents[0]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
 'doc_id': '9e508f2212'}

In [5]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

texts[0]

"Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."

In [6]:
from tqdm.auto import tqdm

In [7]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [8]:
import numpy as np
X = np.array(vectors)
X.shape

(1350, 384)

In [9]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [10]:
scores = X.dot(v_query)

In [11]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [12]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [13]:
top5 = np.argsort(scores)[-5:] # np.argsort sorts from lowest to highest, so the last 5 are the top ones:

In [15]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [16]:
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [17]:
# A trick to reverse the min-sort into a max sort
top5 = np.argsort(-scores)[:5]
top5

array([  2, 625, 907, 538,   7])

In [18]:
# Reading the documents
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 